# Configuration Management Assignment

**Task 1:** Research and compare Ansible, Puppet, and Chef — use cases and advantages.
**Task 2:** Extend the CI pipeline to include continuous delivery (Jenkins/GitLab CI) with a
blue-green deployment strategy.

## Part 1 — Comparing Ansible, Puppet, and Chef

| Aspect | Ansible | Puppet | Chef |
|---|---|---|---|
| Architecture | Agentless (SSH/WinRM) | Agent + master (pull-based) | Agent + server (pull-based) |
| Language | YAML (playbooks) — easy to read | Declarative Puppet DSL (Ruby-like) | Ruby DSL ("recipes"/"cookbooks") |
| Learning curve | Low — YAML is approachable for ops/dev | Medium — DSL + concepts (catalogs, manifests) | Medium/High — full Ruby knowledge helps |
| Push vs Pull | Push (control node initiates) | Pull (agents poll master periodically) | Pull (agents poll server periodically) |
| Idempotency | Yes | Yes | Yes |
| Best for | Ad-hoc automation, orchestration, multi-tier app deployment, teams wanting minimal infra | Large, long-lived infra with strict compliance/state enforcement | Complex, highly customized configs; teams already comfortable with Ruby |
| Scalability | Very good; agentless keeps overhead low | Excellent for thousands of long-running nodes (pull model self-heals) | Excellent, similar to Puppet |
| Windows support | Good (WinRM) | Good | Good |
| Ecosystem | Ansible Galaxy roles | Puppet Forge modules | Chef Supermarket cookbooks |

### Summary of use cases
- **Ansible** — best when you want quick wins, agentless setup, and playbooks that double as
  readable documentation. Great fit for CI/CD-driven infrastructure and application deployment.
- **Puppet** — best for large enterprises needing continuous drift-correction: agents pull
  their desired state periodically and self-heal even without a triggering event.
- **Chef** — best when very fine-grained, programmatic control over configuration logic is
  needed and the team is comfortable writing Ruby.

### Advantages at a glance
- **Ansible:** no agents to maintain, fast to start, huge module library, great for
  orchestrating deployments (not just config).
- **Puppet:** strong reporting/compliance dashboards, mature model for enforcing state at scale.
- **Chef:** highly flexible/programmable, strong testing ecosystem (Test Kitchen, InSpec).

---
## Part 2 — Continuous Delivery with Blue-Green Deployment
We extend a CI pipeline (Jenkins in this example) so that after build + test it performs a **blue-green deployment**: two identical environments (`blue` = currently live, `green` = new version), traffic is switched only after the new version passes a health check, and rollback is instant (just flip traffic back).

### Architecture
```
                 ┌────────────┐
   Users ───────► │   Router/  │
                 │ Load Bal.  │
                 └─────┬──────┘
              (points to ACTIVE color)
         ┌─────────────┴─────────────┐
         ▼                           ▼
   ┌───────────┐               ┌───────────┐
   │   BLUE    │               │   GREEN   │
   │ (current) │               │  (new)    │
   └───────────┘               └───────────┘
```

In [1]:
import os
os.makedirs('project', exist_ok=True)

with open('project/Jenkinsfile', 'w') as f:
    f.write('''pipeline {
    agent any

    environment {
        IMAGE = "myorg/myapp:${env.BUILD_NUMBER}"
    }

    stages {
        stage('Determine active/target color') {
            steps {
                script {
                    try {
                        env.ACTIVE_COLOR = sh(script: "kubectl get service myapp-active -o=jsonpath='{.spec.selector.color}'", returnStdout: true).trim()
                    } catch (Exception e) {
                        env.ACTIVE_COLOR = 'blue' // default fallback if service doesn't exist yet
                    }
                    env.TARGET_COLOR = (env.ACTIVE_COLOR == 'blue') ? 'green' : 'blue'
                    echo "Active: ${env.ACTIVE_COLOR} -> Deploying to: ${env.TARGET_COLOR}"
                }
            }
        }
        stage('Build') {
            steps {
                sh 'docker build -t $IMAGE .'
            }
        }
        stage('Test') {
            steps {
                sh 'docker run --rm $IMAGE npm test'
            }
        }
        stage('Deploy to idle color') {
            steps {
                sh "kubectl set image deployment/myapp-${env.TARGET_COLOR} myapp=$IMAGE"
                sh "kubectl rollout status deployment/myapp-${env.TARGET_COLOR}"
            }
        }
        stage('Smoke test idle color') {
            steps {
                sh "curl -f http://myapp-${env.TARGET_COLOR}-internal/health"
            }
        }
        stage('Switch traffic (blue<->green)') {
            steps {
                sh "kubectl patch service myapp-active -p '{\"spec\":{\"selector\":{\"color\":\"${env.TARGET_COLOR}\"}}}'"
            }
        }
    }

    post {
        failure {
            echo "Deployment failed - traffic remains on ${env.ACTIVE_COLOR}. No rollback needed."
        }
        success {
            echo "Traffic switched to ${env.TARGET_COLOR}. Previous color (${env.ACTIVE_COLOR}) kept warm for instant rollback."
        }
    }
}
''')
print("Created project/Jenkinsfile (blue-green pipeline)")


Created project/Jenkinsfile (blue-green pipeline)


In [2]:
import os
with open('project/k8s-blue-green.yaml', 'w') as f:
    f.write('''# Two deployments (blue/green) + one Service whose selector decides who is "active"
apiVersion: apps/v1
kind: Deployment
metadata:
  name: myapp-blue
spec:
  replicas: 3
  selector:
    matchLabels: { app: myapp, color: blue }
  template:
    metadata:
      labels: { app: myapp, color: blue }
    spec:
      containers:
        - name: myapp
          image: myorg/myapp:stable
          ports: [{ containerPort: 8080 }]
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: myapp-green
spec:
  replicas: 3
  selector:
    matchLabels: { app: myapp, color: green }
  template:
    metadata:
      labels: { app: myapp, color: green }
    spec:
      containers:
        - name: myapp
          image: myorg/myapp:stable
          ports: [{ containerPort: 8080 }]
---
apiVersion: v1
kind: Service
metadata:
  name: myapp-active
spec:
  selector:
    app: myapp
    color: blue        # <- flipped by the pipeline during a release
  ports:
    - port: 80
      targetPort: 8080
''')
print("Created project/k8s-blue-green.yaml")


Created project/k8s-blue-green.yaml


### GitLab CI equivalent (for reference)

In [3]:
gitlab_ci_yaml = '''stages: [build, test, deploy, switch]

build:
  stage: build
  script: docker build -t $CI_REGISTRY_IMAGE:$CI_COMMIT_SHORT_SHA .

test:
  stage: test
  script: docker run --rm $CI_REGISTRY_IMAGE:$CI_COMMIT_SHORT_SHA npm test

deploy_idle:
  stage: deploy
  script:
    - export TARGET_COLOR=$(./scripts/determine-idle-color.sh)
    - kubectl set image deployment/myapp-$TARGET_COLOR myapp=$CI_REGISTRY_IMAGE:$CI_COMMIT_SHORT_SHA
    - kubectl rollout status deployment/myapp-$TARGET_COLOR

switch_traffic:
  stage: switch
  script:
    - kubectl patch service myapp-active -p "{\"spec\":{\"selector\":{\"color\":\"$TARGET_COLOR\"}}}"
  when: manual   # optional manual gate before flipping production traffic
'''
print(gitlab_ci_yaml)


stages: [build, test, deploy, switch]

build:
  stage: build
  script: docker build -t $CI_REGISTRY_IMAGE:$CI_COMMIT_SHORT_SHA .

test:
  stage: test
  script: docker run --rm $CI_REGISTRY_IMAGE:$CI_COMMIT_SHORT_SHA npm test

deploy_idle:
  stage: deploy
  script:
    - export TARGET_COLOR=$(./scripts/determine-idle-color.sh)
    - kubectl set image deployment/myapp-$TARGET_COLOR myapp=$CI_REGISTRY_IMAGE:$CI_COMMIT_SHORT_SHA
    - kubectl rollout status deployment/myapp-$TARGET_COLOR

switch_traffic:
  stage: switch
  script:
    - kubectl patch service myapp-active -p "{"spec":{"selector":{"color":"$TARGET_COLOR"}}}"
  when: manual   # optional manual gate before flipping production traffic

